# Activity 4 — Concurrency in Python

**COM00142M Advanced Programming — Week 7**

**Activity label:** Activity 4 - Concurrency in Python

This notebook implements six classic concurrent programming problems in Python using the `threading` module.

> **Note:** All code runs locally. It will not execute correctly in Google Colab or browser-based environments that do not support native threading.

## Problems covered

1. Update Problem — protecting a critical section with semaphores
2. Bounded Buffer — Producer/Consumer problem
3. Readers and Writers
4. Dining Philosophers — naive (shows deadlock risk) and safe (resource hierarchy)
5. Prime Sieve — concurrent pipeline using `queue.Queue`
6. Chinese Whispers — ring message passing with randomised mutation

In [ ]:
import threading
import queue
import random
import time

---
## 1. Update Problem — Semaphore

Two threads each increment a shared counter 20 times. Without synchronisation, the result is non-deterministic (race condition). With a semaphore protecting the critical section, the result is always 40.

**Critical section:** the read-modify-write of `counter`. A high-level `counter += 1` compiles to three machine instructions (READ, INCREMENT, WRITE), so it is not atomic and requires explicit protection.

In [ ]:
# --- Without semaphore: non-deterministic result (race condition) ---
# Due to Python's GIL, the race may not manifest reliably here,
# but conceptually this is unsynchronised and incorrect design.

counter_unsafe = 0

def increment_unsafe():
    global counter_unsafe
    for _ in range(200_000):      # large N increases chance of visible race
        counter_unsafe += 1

threads_unsafe = [threading.Thread(target=increment_unsafe) for _ in range(2)]
for t in threads_unsafe: t.start()
for t in threads_unsafe: t.join()

print(f"Unsafe counter: {counter_unsafe}  (expected 400000, may differ)")

In [ ]:
# --- With semaphore: guaranteed correct result ---

counter = 0
semaphore = threading.Semaphore(1)  # binary semaphore (mutex)

def increment_safe():
    global counter
    for _ in range(20):
        semaphore.acquire()  # wait (P): blocks if semaphore == 0
        counter += 1         # critical section
        semaphore.release()  # signal (V): increments semaphore, unblocks waiters

threads = [threading.Thread(target=increment_safe) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Safe counter: {counter}  (always 40)")

**Key observations:**
- `Semaphore(1)` is a binary semaphore — only one thread can hold it at a time
- `acquire()` = Dijkstra's P (wait): blocks the caller if the semaphore value is 0
- `release()` = Dijkstra's V (signal): increments the semaphore and unblocks a waiting thread
- Python also provides `threading.Lock()` which is equivalent to `Semaphore(1)` but with ownership semantics

---
## 2. Bounded Buffer — Producer/Consumer

A shared buffer of fixed capacity. Producers add items; consumers remove them. A producer must block when the buffer is full; a consumer must block when it is empty.

**Solution:** `threading.Condition` — a monitor with two condition variables:
- `not_full`: producers wait on this; consumers notify it after removing an item
- `not_empty`: consumers wait on this; producers notify it after adding an item

In [ ]:
BUFFER_SIZE = 5
buffer = []

lock = threading.Lock()
not_full  = threading.Condition(lock)   # producers wait here when buffer is full
not_empty = threading.Condition(lock)   # consumers wait here when buffer is empty


def producer(name, n_items):
    for i in range(n_items):
        with not_full:
            while len(buffer) >= BUFFER_SIZE:
                not_full.wait()                  # block: release lock, wait for space
            item = f"{name}-item{i}"
            buffer.append(item)
            print(f"  {name} produced '{item}'  | buffer: {buffer}")
            not_empty.notify()                   # signal: a consumer can now take
        time.sleep(random.uniform(0.01, 0.05))


def consumer(name, n_items):
    for _ in range(n_items):
        with not_empty:
            while len(buffer) == 0:
                not_empty.wait()                 # block: release lock, wait for items
            item = buffer.pop(0)
            print(f"  {name} consumed '{item}' | buffer: {buffer}")
            not_full.notify()                    # signal: a producer can now place
        time.sleep(random.uniform(0.01, 0.05))


print("--- Bounded Buffer (2 producers × 4 items, 2 consumers × 4 items) ---")
p_threads = [threading.Thread(target=producer, args=(f"P{i}", 4)) for i in range(2)]
c_threads = [threading.Thread(target=consumer, args=(f"C{i}", 4)) for i in range(2)]

for t in p_threads + c_threads: t.start()
for t in p_threads + c_threads: t.join()

print(f"\nFinal buffer (should be empty): {buffer}")

**Key observations:**
- `Condition.wait()` atomically releases the lock and blocks — this is the monitor semantics
- `while` not `if` for the condition check: re-check after wake-up because another thread may have already consumed the item (spurious wakeup protection)
- `notify()` wakes one waiting thread; `notify_all()` wakes all — use `notify_all()` if multiple producers or consumers could be unblocked

---
## 3. Readers and Writers

A shared resource accessed by multiple concurrent threads. Multiple readers may access simultaneously; a writer requires exclusive access.

**Solution (readers-preference):** Track a reader count. The first reader acquires the resource lock (excluding writers); the last reader releases it. Writers simply acquire the resource lock directly.

**Known limitation:** Writer starvation — if readers arrive continuously, writers may never get access.

In [ ]:
shared_data = 0
readers_count = 0

resource_lock = threading.Lock()   # controls exclusive access to shared_data
readers_lock  = threading.Lock()   # controls access to readers_count


def reader(name, n_reads):
    global readers_count, shared_data
    for _ in range(n_reads):
        # --- Enter read section ---
        with readers_lock:
            readers_count += 1
            if readers_count == 1:       # first reader locks out writers
                resource_lock.acquire()

        print(f"  {name} reads: {shared_data}  ({readers_count} reader(s) active)")
        time.sleep(random.uniform(0.005, 0.02))

        # --- Exit read section ---
        with readers_lock:
            readers_count -= 1
            if readers_count == 0:       # last reader releases the resource
                resource_lock.release()

        time.sleep(random.uniform(0.005, 0.01))


def writer(name, increment, n_writes):
    global shared_data
    for _ in range(n_writes):
        with resource_lock:              # exclusive access: blocks all readers and other writers
            shared_data += increment
            print(f"  {name} writes: {shared_data}")
        time.sleep(random.uniform(0.01, 0.03))


print("--- Readers and Writers (2 readers × 4 reads, 2 writers × 2 writes) ---")
shared_data = 0
readers_count = 0

rw_threads = [
    threading.Thread(target=reader, args=("Reader-1", 4)),
    threading.Thread(target=reader, args=("Reader-2", 4)),
    threading.Thread(target=writer, args=("Writer-1", 10, 2)),
    threading.Thread(target=writer, args=("Writer-2",  5, 2)),
]
for t in rw_threads: t.start()
for t in rw_threads: t.join()

print(f"\nFinal shared_data: {shared_data}  (expected 30: 2×10 + 2×5)")

**Key observations:**
- Two locks: `readers_lock` protects `readers_count`; `resource_lock` protects `shared_data`
- Multiple readers can read simultaneously (no resource lock between first and last reader)
- Writers always acquire resource lock exclusively
- **Starvation risk:** If readers continuously arrive before `readers_count` reaches 0, writers starve

---
## 4. Dining Philosophers

Five philosophers at a circular table. One fork between each adjacent pair. Each philosopher needs both adjacent forks to eat.

**Naive version** (pick up left fork, then right): all philosophers can pick up their left fork simultaneously → circular wait → **deadlock**.

**Safe version** (resource hierarchy): always acquire the lower-numbered fork first. This breaks circular wait — no deadlock possible.

In [ ]:
# --- Naive version: demonstrates deadlock risk ---
# NOTE: may deadlock. Run with a timeout for safety.
# Uncomment to observe deadlock behaviour. Commented here to avoid hanging the notebook.

# N = 5
# forks_naive = [threading.Lock() for _ in range(N)]
# 
# def philosopher_naive(i, n_meals=3):
#     left = i
#     right = (i + 1) % N
#     for _ in range(n_meals):
#         time.sleep(random.uniform(0.001, 0.01))  # think
#         forks_naive[left].acquire()              # pick up left fork
#         forks_naive[right].acquire()             # pick up right fork -- may deadlock!
#         print(f"  Philosopher {i} eating")
#         forks_naive[right].release()
#         forks_naive[left].release()
# 
# threads = [threading.Thread(target=philosopher_naive, args=(i,)) for i in range(N)]
# for t in threads: t.start()
# for t in threads: t.join(timeout=3)  # 3s timeout to detect deadlock
# print([t.is_alive() for t in threads])  # True = deadlocked

print("Naive philosopher code is commented out to prevent deadlock hanging the notebook.")
print("Deadlock condition: all 5 philosophers acquire left fork simultaneously → circular wait.")

In [ ]:
# --- Safe version: resource hierarchy (always acquire lower-numbered fork first) ---

N = 5
forks = [threading.Semaphore(1) for _ in range(N)]
meals_eaten = [0] * N
print_lock = threading.Lock()


def philosopher(i, n_meals=3):
    left  = i
    right = (i + 1) % N

    # Resource hierarchy: always pick up the lower-numbered fork first.
    # This breaks circular wait — Philosopher 4's 'first' fork is fork 0 (not fork 4),
    # so they compete with Philosopher 0 for fork 0. No cycle forms.
    first, second = (left, right) if left < right else (right, left)

    for meal in range(n_meals):
        # Think
        time.sleep(random.uniform(0.001, 0.01))

        # Pick up forks in safe order
        forks[first].acquire()
        forks[second].acquire()

        # Eat
        meals_eaten[i] += 1
        with print_lock:
            print(f"  Philosopher {i} eating (meal {meal + 1}) "
                  f"| forks {first} & {second}")

        forks[second].release()
        forks[first].release()


print("--- Dining Philosophers (resource hierarchy — deadlock-free) ---")
phil_threads = [threading.Thread(target=philosopher, args=(i,)) for i in range(N)]
for t in phil_threads: t.start()
for t in phil_threads: t.join()

print(f"\nMeals per philosopher: {meals_eaten}  (3 each expected)")
print(f"Which problems can this scenario generate?")
print(f"  - Deadlock:    YES (naive version; resource hierarchy prevents it)")
print(f"  - Livelock:    YES (if using polite retry with synchronised back-off)")
print(f"  - Starvation:  YES (unfair scheduling can prevent one philosopher from ever eating)")
print(f"  - Race condition: YES (unprotected access to shared forks in naive version)")

**Resource hierarchy explanation:**
- Forks are numbered 0–4
- Each philosopher always picks up `min(left, right)` first, then `max(left, right)`
- Philosopher 4: `left=4, right=0` → picks up fork **0** first (since 0 < 4)
- This breaks the circular wait: Phil 4 and Phil 0 both compete for fork 0, so one blocks. No cycle.
- Coffman's circular-wait condition is violated → deadlock is impossible

---
## 5. Prime Sieve — Concurrent Pipeline

The Sieve of Eratosthenes implemented as a concurrent pipeline using message passing via `queue.Queue`.

**Design:**
- One generator thread produces candidate numbers into a queue
- Each discovered prime spawns a new filter thread for that prime
- Each filter thread reads from its input queue, discards multiples of its prime, and passes the rest to the next queue
- The first number each stage receives is, by definition, prime (it passed all upstream filters)

In [ ]:
def generate(out_q, limit):
    """Send integers 2..limit into the first queue, then a None sentinel."""
    for i in range(2, limit + 1):
        out_q.put(i)
    out_q.put(None)  # sentinel: signals end of stream


def filter_stage(in_q, out_q, prime):
    """Pass numbers from in_q to out_q, dropping multiples of prime."""
    while True:
        n = in_q.get()
        if n is None:
            out_q.put(None)  # propagate sentinel downstream
            return
        if n % prime != 0:   # not a multiple of this prime — pass on
            out_q.put(n)


def prime_sieve(limit):
    """
    Concurrent prime sieve using a thread pipeline.
    Returns all primes up to limit.
    """
    primes = []
    gen_q = queue.Queue()

    # Start the number generator
    gen_thread = threading.Thread(target=generate, args=(gen_q, limit))
    gen_thread.start()

    current_q = gen_q

    # Main thread acts as the collector and pipeline builder
    while True:
        n = current_q.get()
        if n is None:
            break  # sentinel received: all primes found

        # n is prime (it passed all upstream filters)
        primes.append(n)

        # Spawn a new filter stage for this prime
        next_q = queue.Queue()
        t = threading.Thread(
            target=filter_stage,
            args=(current_q, next_q, n),
            daemon=True
        )
        t.start()
        current_q = next_q

    gen_thread.join()
    return primes


primes_50 = prime_sieve(50)
print(f"Primes up to 50:  {primes_50}")
print(f"Count: {len(primes_50)}")

primes_100 = prime_sieve(100)
print(f"\nPrimes up to 100: {primes_100}")
print(f"Count: {len(primes_100)}")

**How termination works:**
- Generator sends `None` as a sentinel at the end of the number stream
- Each filter stage, on receiving `None`, propagates it downstream and exits
- The main thread exits its loop when the final stage outputs `None`

**Terminating after K primes (variant):** Replace the `while True` loop with a counter:
```python
while len(primes) < k:
    n = current_q.get()
    ...
```

**Limitation:** One thread per prime — for large limits (e.g., primes up to 10,000 → 1,229 threads), the thread overhead becomes significant. Not practical for large-scale prime generation; useful as a teaching example of concurrent pipelines.

---
## 6. Chinese Whispers

A circular ring of threads. The starting thread sends a message; each thread optionally corrupts it and passes it on; when the message returns to the starter, original and final versions are compared.

**Demonstrates:** ring-topology message passing, asynchronous communication via `queue.Queue`, and termination propagation via a sentinel.

In [ ]:
def introduce_error(msg):
    """Swap two random characters in the message."""
    if len(msg) < 2:
        return msg
    msg_list = list(msg)
    i, j = random.sample(range(len(msg_list)), 2)
    msg_list[i], msg_list[j] = msg_list[j], msg_list[i]
    return ''.join(msg_list)


def player(pid, in_q, out_q, is_starter, original_msg, error_chance, result_holder):
    """
    Each player receives a message, possibly corrupts it, and forwards it.
    The starter player also sends the initial message and records the final result.
    """
    if is_starter:
        # Send the original message into the ring
        out_q.put(original_msg)

        # Wait for the message to travel the full ring and return
        received = in_q.get()
        result_holder['final'] = received
        result_holder['match'] = (received == original_msg)

        # Send termination sentinel around the ring to clean up all players
        out_q.put(None)

    else:
        while True:
            msg = in_q.get()

            if msg is None:              # termination sentinel
                out_q.put(None)
                return

            # Randomly decide to introduce an error
            if random.random() < error_chance:
                original = msg
                msg = introduce_error(msg)
                print(f"  Player {pid}: '{original}' → '{msg}'  [error introduced]")
            else:
                print(f"  Player {pid}: '{msg}'  [passed unchanged]")

            out_q.put(msg)


def chinese_whispers(original_message, n_players=7, error_chance=0.4):
    # Create N queues — one per link between players
    # Player i reads from queues[i] and writes to queues[(i+1) % N]
    queues = [queue.Queue() for _ in range(n_players)]
    result_holder = {}

    threads = []
    for i in range(n_players):
        in_q  = queues[i]
        out_q = queues[(i + 1) % n_players]  # circular ring
        t = threading.Thread(
            target=player,
            args=(i, in_q, out_q, i == 0, original_message, error_chance, result_holder)
        )
        threads.append(t)

    for t in threads: t.start()
    for t in threads: t.join()

    return result_holder


print("--- Chinese Whispers (7 players, 40% error chance) ---")
result = chinese_whispers(
    original_message="Hello World",
    n_players=7,
    error_chance=0.4
)
print(f"\nOriginal: 'Hello World'")
print(f"Final:    '{result['final']}'")
print(f"Messages match: {result['match']}")

In [ ]:
# Run with different error rates to observe the effect
print("--- Effect of error_chance on message corruption ---")
for rate in [0.0, 0.2, 0.5, 1.0]:
    outcomes = []
    for _ in range(10):
        r = chinese_whispers("Hello World", n_players=5, error_chance=rate)
        outcomes.append(r['match'])
    match_rate = sum(outcomes) / len(outcomes)
    print(f"  error_chance={rate:.1f}: message survived intact {match_rate*100:.0f}% of the time")

**Key observations:**
- This is **indirect asymmetric naming**: each player only knows the next player's queue, not who sent to them
- `queue.Queue` is thread-safe by design — no explicit locking needed
- Termination propagates as a sentinel (`None`) around the full ring
- At `error_chance=0.0`, the message always survives intact; at `1.0`, every player mutates it